# 📊 Analisi Benchmark — Flight Delay 2024

Questo notebook analizza i risultati dei benchmark eseguiti su tutte e 3 le analisi
(3.1 Statistiche Compagnie, 3.2 Report Ritardi, 3.3 Ranking Anomalo)
con 4 tecnologie (MapReduce, Hive, Spark Core, Spark SQL)
su 6 dimensioni di dataset (10%, 25%, 50%, 100%, 125%, 150%).

**Totale benchmark**: 54 esecuzioni (9 job × 6 dimensioni)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import os

PLOTS_DIR = 'plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

# Stile globale
plt.rcParams.update({
    'figure.dpi': 150, 'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa', 'axes.grid': True, 'grid.alpha': 0.3,
    'font.size': 11, 'axes.titlesize': 14, 'axes.labelsize': 12,
    'legend.fontsize': 10, 'figure.titlesize': 16,
})

# Palette
COLORS = {'mapreduce': '#e74c3c', 'hive': '#f39c12', 'spark_core': '#2ecc71', 'spark_sql': '#3498db'}
LABELS = {'mapreduce': 'MapReduce', 'hive': 'Hive', 'spark_core': 'Spark Core', 'spark_sql': 'Spark SQL'}
MARKERS = {'mapreduce': 's', 'hive': 'D', 'spark_core': '^', 'spark_sql': 'o'}
A_LABELS = {'3.1': '3.1 — Statistiche Compagnie', '3.2': '3.2 — Report Ritardi', '3.3': '3.3 — Ranking Anomalo'}
SAMPLE_ORDER = ['sample_010pct','sample_025pct','sample_050pct','full_dataset','sample_125pct','sample_150pct']
SAMPLE_LABELS = ['10%','25%','50%','100%','125%','150%']

print('Setup completato ✅')

## 1. Caricamento Dati

In [ ]:
df = pd.read_csv('results_local.csv')
df['analysis'] = df['analysis'].astype(str)
df['notes'] = df['notes'].fillna('full_dataset')
df['sample_order'] = df['notes'].map({s: i for i, s in enumerate(SAMPLE_ORDER)})
df = df.sort_values(['analysis', 'technology', 'sample_order'])

print(f'Totale benchmark: {len(df)}')
print(f'Analisi: {sorted(df["analysis"].unique())}')
print(f'Tecnologie: {sorted(df["technology"].unique())}')
print(f'\nDataset completo (100%):')
display(df[df['notes']=='full_dataset'][['analysis','technology','input_rows','elapsed_sec']].reset_index(drop=True))

## 2. Tabella Riepilogativa — Dataset Completo

In [ ]:
full = df[df['notes']=='full_dataset'].copy()
pivot = full.pivot_table(index='analysis', columns='technology', values='elapsed_sec', aggfunc='first')
pivot.index = [A_LABELS.get(a, a) for a in pivot.index]
pivot.columns = [LABELS.get(t, t) for t in pivot.columns]
pivot = pivot[['MapReduce','Hive','Spark Core','Spark SQL']]
display(pivot.style.format('{:.2f}s').highlight_min(axis=1, color='#d4edda').highlight_max(axis=1, color='#f8d7da'))

## 3. Confronto Tempi — Dataset Completo (7.04M righe)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
analyses = sorted(full['analysis'].unique())
x = np.arange(len(analyses))
width = 0.18
offsets = {'mapreduce': -1.5, 'hive': -0.5, 'spark_core': 0.5, 'spark_sql': 1.5}

for tech, offset in offsets.items():
    subset = full[full['technology']==tech]
    if subset.empty: continue
    times = [subset[subset['analysis']==a]['elapsed_sec'].values[0] if not subset[subset['analysis']==a].empty else 0 for a in analyses]
    bars = ax.bar(x + offset*width, times, width, label=LABELS[tech], color=COLORS[tech], edgecolor='white', linewidth=0.5)
    for bar, t in zip(bars, times):
        if t > 0: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2, f'{t:.1f}s', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xlabel('Analisi'); ax.set_ylabel('Tempo (secondi)')
ax.set_title('Confronto Tempi — Dataset Completo (7.04M righe)')
ax.set_xticks(x); ax.set_xticklabels([A_LABELS.get(a,a) for a in analyses])
ax.legend(loc='upper left'); ax.set_ylim(0, max(full['elapsed_sec'])*1.2)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/01_full_dataset_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Scalabilità per Analisi

In [ ]:
for analysis in sorted(df['analysis'].unique()):
    adf = df[df['analysis']==analysis]
    fig, ax = plt.subplots(figsize=(10, 6))
    for tech in sorted(adf['technology'].unique()):
        tdf = adf[adf['technology']==tech].sort_values('sample_order')
        ax.plot(tdf['input_rows']/1e6, tdf['elapsed_sec'], marker=MARKERS.get(tech,'o'),
                color=COLORS.get(tech,'#666'), label=LABELS.get(tech,tech), linewidth=2, markersize=8)
    ax.set_xlabel('Dimensione input (milioni di righe)'); ax.set_ylabel('Tempo (secondi)')
    ax.set_title(f'Scalabilità — Analisi {A_LABELS.get(analysis, analysis)}')
    ax.legend()
    num = {'3.1':'02','3.2':'03','3.3':'04'}[analysis]
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/{num}_scalability_{analysis.replace(".","")}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Heatmap Tempi (Tecnologia × Analisi)

In [ ]:
techs = ['mapreduce','hive','spark_core','spark_sql']
analyses = sorted(full['analysis'].unique())
matrix = np.array([[full[(full['technology']==t)&(full['analysis']==a)]['elapsed_sec'].values[0] if not full[(full['technology']==t)&(full['analysis']==a)].empty else np.nan for a in analyses] for t in techs])

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(analyses))); ax.set_xticklabels([A_LABELS.get(a,a) for a in analyses])
ax.set_yticks(range(len(techs))); ax.set_yticklabels([LABELS[t] for t in techs])
for i in range(len(techs)):
    for j in range(len(analyses)):
        val = matrix[i,j]
        if not np.isnan(val):
            color = 'white' if val > matrix[~np.isnan(matrix)].mean() else 'black'
            ax.text(j, i, f'{val:.1f}s', ha='center', va='center', fontweight='bold', fontsize=11, color=color)
        else:
            ax.text(j, i, '—', ha='center', va='center', fontsize=11, color='#999')
ax.set_title('Heatmap Tempi — Dataset Completo (7.04M righe)')
fig.colorbar(im, ax=ax, label='Tempo (secondi)')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/05_heatmap_full.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Speedup Relativo

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
analyses = sorted(full['analysis'].unique())
x = np.arange(len(analyses)); width = 0.18
offsets = {'mapreduce': -1.5, 'hive': -0.5, 'spark_core': 0.5, 'spark_sql': 1.5}
for tech, offset in offsets.items():
    speedups = []
    for a in analyses:
        ta = full[full['analysis']==a]; max_t = ta['elapsed_sec'].max()
        tt = ta[ta['technology']==tech]['elapsed_sec']
        speedups.append(max_t/tt.values[0] if not tt.empty else 0)
    if any(s>0 for s in speedups):
        bars = ax.bar(x+offset*width, speedups, width, label=LABELS[tech], color=COLORS[tech], edgecolor='white')
        for bar, s in zip(bars, speedups):
            if s>0: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1, f'{s:.1f}×', ha='center', fontsize=8, fontweight='bold')
ax.set_xlabel('Analisi'); ax.set_ylabel('Speedup (vs più lenta)')
ax.set_title('Speedup Relativo — Dataset Completo')
ax.set_xticks(x); ax.set_xticklabels([A_LABELS.get(a,a) for a in analyses])
ax.axhline(y=1, color='#e74c3c', linestyle='--', alpha=0.5, label='Baseline (1×)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/06_speedup.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Confronto Formato — CSV vs Parquet

In [ ]:
csv_sizes = df[df['input_file'].str.endswith('.csv')].groupby('notes')['input_size_mb'].first()
pq_sizes = df[df['input_file'].str.endswith('.parquet')].groupby('notes')['input_size_mb'].first()
common = sorted(set(csv_sizes.index)&set(pq_sizes.index), key=lambda s: SAMPLE_ORDER.index(s) if s in SAMPLE_ORDER else 99)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(common)); w = 0.35
csv_v = [csv_sizes.get(s,0) for s in common]; pq_v = [pq_sizes.get(s,0) for s in common]
ax.bar(x-w/2, csv_v, w, label='CSV', color='#e74c3c', alpha=0.8)
ax.bar(x+w/2, pq_v, w, label='Parquet', color='#3498db', alpha=0.8)
for i, (c,p) in enumerate(zip(csv_v, pq_v)):
    if p>0: ax.text(x[i]+w/2, p+5, f'{c/p:.1f}×', ha='center', fontsize=9, fontweight='bold')
labels = [SAMPLE_LABELS[SAMPLE_ORDER.index(s)] if s in SAMPLE_ORDER else s for s in common]
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_xlabel('Dimensione dataset'); ax.set_ylabel('Dimensione file (MB)')
ax.set_title('Confronto Formato — CSV vs Parquet'); ax.legend()
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/07_format_csv_vs_parquet.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Overhead Hive — Avvio vs Elaborazione

In [ ]:
hive = df[df['technology']=='hive']
fig, axes = plt.subplots(1, len(hive['analysis'].unique()), figsize=(15, 5), sharey=True)
for ax, analysis in zip(axes, sorted(hive['analysis'].unique())):
    adf = hive[hive['analysis']==analysis].sort_values('sample_order')
    times = adf['elapsed_sec'].values
    overhead = times.min() * 0.9
    variable = times - overhead
    ax.bar(range(len(times)), [overhead]*len(times), color='#f39c12', alpha=0.4, label='Overhead fisso')
    ax.bar(range(len(times)), np.maximum(variable,0), bottom=overhead, color='#f39c12', alpha=0.8, label='Elaborazione')
    ax.set_xticks(range(len(times))); ax.set_xticklabels(SAMPLE_LABELS[:len(times)], rotation=45)
    ax.set_title(f'Analisi {analysis}'); ax.set_xlabel('Dimensione')
    if ax==axes[0]: ax.set_ylabel('Tempo (secondi)')
axes[0].legend(loc='upper left', fontsize=9)
fig.suptitle('Hive — Overhead di Avvio vs Elaborazione Effettiva', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/08_hive_overhead.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Throughput (righe/secondo)

In [ ]:
full_t = full.copy(); full_t['throughput'] = full_t['input_rows']/full_t['elapsed_sec']
fig, ax = plt.subplots(figsize=(10, 6))
analyses = sorted(full_t['analysis'].unique())
x = np.arange(len(analyses)); width = 0.18
offsets = {'mapreduce': -1.5, 'hive': -0.5, 'spark_core': 0.5, 'spark_sql': 1.5}
for tech, offset in offsets.items():
    subset = full_t[full_t['technology']==tech]
    if subset.empty: continue
    tp = [subset[subset['analysis']==a]['throughput'].values[0]/1000 if not subset[subset['analysis']==a].empty else 0 for a in analyses]
    bars = ax.bar(x+offset*width, tp, width, label=LABELS[tech], color=COLORS[tech], edgecolor='white')
    for bar, t in zip(bars, tp):
        if t>0: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5, f'{t:.0f}K', ha='center', fontsize=8, fontweight='bold')
ax.set_xlabel('Analisi'); ax.set_ylabel('Throughput (K righe/s)')
ax.set_title('Throughput — Dataset Completo (7.04M righe)')
ax.set_xticks(x); ax.set_xticklabels([A_LABELS.get(a,a) for a in analyses])
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/09_throughput.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Panoramica Scalabilità Unificata

In [ ]:
analyses = sorted(df['analysis'].unique())
fig, axes = plt.subplots(1, len(analyses), figsize=(18, 5), sharey=False)
for ax, analysis in zip(axes, analyses):
    adf = df[df['analysis']==analysis]
    for tech in sorted(adf['technology'].unique()):
        tdf = adf[adf['technology']==tech].sort_values('sample_order')
        ax.plot(tdf['input_rows']/1e6, tdf['elapsed_sec'], marker=MARKERS.get(tech,'o'),
                color=COLORS.get(tech,'#666'), label=LABELS.get(tech,tech), linewidth=2, markersize=7)
    ax.set_title(f'Analisi {analysis}', fontweight='bold')
    ax.set_xlabel('Milioni di righe')
    if ax==axes[0]: ax.set_ylabel('Tempo (secondi)')
    ax.legend(fontsize=9)
fig.suptitle('Scalabilità — Tutte le Analisi', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/10_scalability_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(f'\n✅ Tutti i grafici salvati in: {PLOTS_DIR}/')
print(f'   Grafici generati: {len(os.listdir(PLOTS_DIR))}')